# Exact Optimization Model (MIP)

## Purpose
Find the **exact optimal tote entry order and item placement order** that minimises the sum of order completion times while rewarding flexible sequencing (optionality). An order is considered complete when the sortation machine has finished processing all items in that order — i.e., when the last tote belonging to that order has been fully fed into the sorter.

## Decision Variables

**Tote sequencing:**
- `x[i,j] ∈ {0,1}`: 1 if tote `i` is immediately followed by tote `j` in the sequence
- `s[i] ∈ {0,1}`: 1 if tote `i` is the first tote in the sequence
- `e[i] ∈ {0,1}`: 1 if tote `i` is the last tote in the sequence
- `u[i]` integer: position of tote `i` in the sequence (1..n), used for MTZ subtour elimination and optionality position weighting
- `F[j] ≥ 0`: cumulative finish time of tote `j` on the sorter (time when the last item from tote `j` has been placed)
- `W[i,j] ≥ 0`: linearisation auxiliary — represents `x[i,j] × F[i]`
- `C[k] ≥ 0`: completion time of order `k` (finish time of order `k`'s last tote in the sequence)

**Item sequencing** (solved independently per tote after tote order is fixed):
- Same `x`, `s`, `e`, `u`, `F`, `W` structure at the item level within each tote
- `C[k] ≥ 0` for orders completing at this tote: local finish time when their last item is placed

## Objective
Minimise the composite score:

**sum of order completion times − λ × optionality score**

where:
- `sum C[k]` — total time summed across all orders from start until each order's last item is sorted
- `λ = 0.35` (OPTIONALITY_LAMBDA)
- Optionality score = position-weighted node reward (totes with many compatible successors placed earlier) + edge bonus for compatible tote-to-tote transitions (no bin switch required)

## Constraints

**Sequencing flow:**
- Each tote has exactly one incoming arc, or is the start tote
- Each tote has exactly one outgoing arc, or is the end tote
- Exactly one start tote and one end tote

**Subtour elimination (MTZ):**
- `u[i] − u[j] + n·x[i,j] ≤ n − 1` for all arcs (i,j) — prevents disconnected sub-cycles

**Finish time propagation:**
- `F[j] = fixed_block_cost(j) + Σᵢ (W[i,j] + edge_cost(i,j)·x[i,j])`
- For the start tote all incoming arc weights are zero, so `F[start] = fixed_block_cost(start)` automatically

**Bilinear linearisation of W[i,j] = x[i,j] × F[i]:**
- `W[i,j] ≤ F_max · x[i,j]`
- `W[i,j] ≥ F[i] − F_max · (1 − x[i,j])`
- `W[i,j] ≤ F[i]`

**Order completion:**
- `C[k] ≥ F[t]` for every tote `t` belonging to order `k` — forces `C[k]` to equal the finish time of order `k`'s last tote

## Cost Parameters
- `PLACE_TIME = 1.75s` per item placed onto the sorter
- `TOTE_SWITCH_TIME = 4.0s` penalty when moving between any two consecutive totes
- `BIN_SWITCH_TIME = 0.75s` additional penalty when consecutive totes (or items) belong to different order bins

In [1]:
import csv
from pathlib import Path
import gurobipy as gp
from gurobipy import GRB

# Choose which generated input run(s) to use.
# - RUN_ID = None  -> canonical inputs/
# - RUN_ID = int   -> one run folder inputs/runs/run_XXXX
# - RUN_ID = "all" -> all run folders under inputs/runs/
RUN_ID = "all"

MIP_TIME_LIMIT = 30  # seconds per solve — big-M formulation is slow to prove optimality;
                     # 30s is sufficient to find a near-optimal incumbent on these problem sizes

def _resolve_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return paths[0]


def _get_input_bases(run_id):
    if run_id == "all":
        runs_root = _resolve_existing([Path("inputs/runs"), Path("../inputs/runs")])
        if not runs_root.exists():
            return []
        return sorted([p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith("run_")])
    if run_id is None:
        return [_resolve_existing([Path("inputs"), Path("../inputs")])]
    run_name = f"run_{int(run_id):04d}"
    return [_resolve_existing([Path("inputs/runs") / run_name, Path("../inputs/runs") / run_name])]


output_base = _resolve_existing([Path("outputs"), Path("../outputs")])
output_base.mkdir(parents=True, exist_ok=True)
OUT = output_base

NUM_CONVEYORS = 4
PLACE_TIME = 1.75
TOTE_SWITCH_TIME = 4.0
BIN_SWITCH_TIME = 0.75

# Composite objective = sum_order_completion_times - OPTIONALITY_LAMBDA * optionality_score
OPTIONALITY_LAMBDA = 0.35
OPT_EDGE_WEIGHT = 1.0
OPT_BRANCH_WEIGHT = 0.6
OPT_RARE_WEIGHT = 0.4


def _coerce_int(v):
    s = v.strip()
    if s == "":
        return None
    try:
        return int(float(s))
    except ValueError:
        return None


def _read_rows(p):
    with p.open("r", newline="") as f:
        return list(csv.reader(f))


def build_tasks():
    item_rows = _read_rows(INPUT_ITEMTYPES)
    qty_rows = _read_rows(INPUT_QUANTITIES)
    tote_rows = _read_rows(INPUT_TOTES)
    n_orders = max(len(item_rows), len(qty_rows), len(tote_rows))

    tasks = []
    for i in range(n_orders):
        ir = item_rows[i] if i < len(item_rows) else []
        qr = qty_rows[i] if i < len(qty_rows) else []
        tr = tote_rows[i] if i < len(tote_rows) else []
        width = max(len(ir), len(qr), len(tr))
        for j in range(width):
            item_type = _coerce_int(ir[j]) if j < len(ir) else None
            qty = _coerce_int(qr[j]) if j < len(qr) else None
            tote = _coerce_int(tr[j]) if j < len(tr) else None
            if item_type is None or qty is None or tote is None or qty <= 0:
                continue
            tasks.append({"order_id": i + 1, "item_type": item_type, "tote": tote, "qty": qty})
    return tasks


def build_tote_blocks(tasks):
    tote_to_bins = {}
    tote_to_items = {}
    for t in tasks:
        tote_to_bins.setdefault(t["tote"], []).extend([t["order_id"]] * t["qty"])
        tote_to_items.setdefault(t["tote"], []).extend([t["item_type"]] * t["qty"])

    blocks = {}
    for tote, bins in tote_to_bins.items():
        seq = sorted(bins)
        internal_bin_switches = sum(1 for k in range(1, len(seq)) if seq[k] != seq[k - 1])
        blocks[tote] = {
            "first_bin": seq[0],
            "last_bin": seq[-1],
            "units": len(seq),
            "internal_bin_switches": internal_bin_switches,
            "items": tote_to_items[tote],
            "item_pairs": list(zip(bins, tote_to_items[tote])),
        }
    return blocks


def build_order_tote_map(blocks):
    """Map each order_id to the set of totes that contain its items."""
    order_tote_map = {}
    for tote, block in blocks.items():
        for order_id, _ in block["item_pairs"]:
            order_tote_map.setdefault(order_id, set()).add(tote)
    return order_tote_map


def edge_cost(i, j, blocks):
    c = TOTE_SWITCH_TIME
    if blocks[i]["last_bin"] != blocks[j]["first_bin"]:
        c += BIN_SWITCH_TIME
    return c


def fixed_block_cost(i, blocks):
    b = blocks[i]
    return b["units"] * PLACE_TIME + b["internal_bin_switches"] * BIN_SWITCH_TIME


def build_optionality_terms(blocks):
    totes = sorted(blocks.keys())
    first_bins = {t: blocks[t]["first_bin"] for t in totes}
    last_bins = {t: blocks[t]["last_bin"] for t in totes}

    compat = set()
    branch = {t: 0 for t in totes}

    for i in totes:
        for j in totes:
            if i == j:
                continue
            if last_bins[i] == first_bins[j]:
                compat.add((i, j))
                branch[i] += 1

    bin_freq = {}
    for t in totes:
        b = first_bins[t]
        bin_freq[b] = bin_freq.get(b, 0) + 1

    node_coeff = {}
    for t in totes:
        rarity = 1.0 / bin_freq[first_bins[t]]
        node_coeff[t] = OPT_BRANCH_WEIGHT * branch[t] - OPT_RARE_WEIGHT * rarity

    return {"compat": compat, "node_coeff": node_coeff}


def build_item_optionality_terms(item_pairs):
    """Build optionality terms for item sequencing within a tote.
    Items are indexed 0..n-1. Two items are compatible (no bin switch) when
    they share the same order_id."""
    n = len(item_pairs)
    order_ids = [p[0] for p in item_pairs]

    compat = set()
    branch = [0] * n
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if order_ids[i] == order_ids[j]:
                compat.add((i, j))
                branch[i] += 1

    freq = {}
    for oid in order_ids:
        freq[oid] = freq.get(oid, 0) + 1

    node_coeff = {}
    for i, (oid, _) in enumerate(item_pairs):
        node_coeff[i] = OPT_BRANCH_WEIGHT * branch[i] - OPT_RARE_WEIGHT / freq[oid]

    return {"compat": compat, "node_coeff": node_coeff}


def sequence_metrics(seq, blocks, terms, order_tote_map):
    """Returns (total_time, sum_completion, optionality, objective).

    sum_completion = sum over all orders of the finish time of that order's
    last tote in the sequence (i.e., when the sortation machine finishes
    processing that tote, the order is considered complete).
    """
    n = len(seq)
    cumulative = 0.0
    tote_finish = {}
    optionality = 0.0

    prev = None
    for pos, tote in enumerate(seq, start=1):
        cumulative += fixed_block_cost(tote, blocks)
        if prev is not None:
            cumulative += edge_cost(prev, tote, blocks)
            if (prev, tote) in terms["compat"]:
                optionality += OPT_EDGE_WEIGHT
        tote_finish[tote] = cumulative
        optionality += terms["node_coeff"][tote] * (n + 1 - pos)
        prev = tote

    total_time = cumulative
    seq_set = set(seq)
    sum_completion = 0.0
    for order_id, totes in order_tote_map.items():
        in_seq = [t for t in totes if t in seq_set]
        if in_seq:
            last_tote = max(in_seq, key=lambda t: tote_finish[t])
            sum_completion += tote_finish[last_tote]

    objective = sum_completion - OPTIONALITY_LAMBDA * optionality
    return total_time, sum_completion, optionality, objective


def solve_exact_mip(blocks, terms, order_tote_map):
    totes = sorted(blocks.keys())
    n = len(totes)

    if n == 0:
        return [], 0.0, "empty"

    arcs = [(i, j) for i in totes for j in totes if i != j]
    orders = list(order_tote_map.keys())

    # Upper bound on any tote's finish time
    F_max = sum(fixed_block_cost(t, blocks) for t in totes) + (n - 1) * (TOTE_SWITCH_TIME + BIN_SWITCH_TIME)

    model = gp.Model("tote_sequence_exact")
    model.Params.OutputFlag = 0
    model.Params.TimeLimit = MIP_TIME_LIMIT

    x = model.addVars(arcs, vtype=GRB.BINARY, name="x")
    s = model.addVars(totes, vtype=GRB.BINARY, name="s")
    e = model.addVars(totes, vtype=GRB.BINARY, name="e")
    # u[i] encodes position (1..n) — used for MTZ subtour elimination and optionality position weight
    u = model.addVars(totes, vtype=GRB.INTEGER, lb=1, ub=n, name="u")

    # F[j] = cumulative finish time of tote j on the sorter
    F = model.addVars(totes, lb=0.0, ub=F_max, name="F")
    # W[i,j] linearizes x[i,j] * F[i] to handle the bilinear finish-time propagation
    W = model.addVars(arcs, lb=0.0, ub=F_max, name="W")
    # C[k] = completion time for order k (finish time of order k's last tote in the sequence)
    C = model.addVars(orders, lb=0.0, ub=F_max, name="C")

    n_float = float(n)

    # Optionality terms (unchanged — position-weighted node reward + edge bonus)
    node_optional = gp.quicksum(
        terms["node_coeff"][i] * (n_float + 1.0 - u[i]) for i in totes
    )
    compat_optional = gp.quicksum(
        OPT_EDGE_WEIGHT * x[i, j] for (i, j) in terms["compat"]
    )

    model.setObjective(
        gp.quicksum(C[k] for k in orders) - OPTIONALITY_LAMBDA * (node_optional + compat_optional),
        GRB.MINIMIZE,
    )

    # Flow constraints: each tote has exactly one predecessor (or is start) and one successor (or is end)
    for i in totes:
        model.addConstr(gp.quicksum(x[j, i] for j in totes if j != i) + s[i] == 1)
    for i in totes:
        model.addConstr(gp.quicksum(x[i, j] for j in totes if j != i) + e[i] == 1)
    model.addConstr(gp.quicksum(s[i] for i in totes) == 1)
    model.addConstr(gp.quicksum(e[i] for i in totes) == 1)

    # MTZ subtour elimination
    for i, j in arcs:
        model.addConstr(u[i] - u[j] + n * x[i, j] <= n - 1)

    # Tote finish time propagation:
    # F[j] = fixed_block_cost(j) + sum_i( x[i,j]*(F[i] + edge_cost(i,j)) )
    #       = fixed_block_cost(j) + sum_i( W[i,j] + edge_cost(i,j)*x[i,j] )
    # For the start tote: sum_i x[i,j] = 0, so F[j] = fixed_block_cost(j) automatically.
    for j in totes:
        model.addConstr(
            F[j] == fixed_block_cost(j, blocks) + gp.quicksum(
                W[i, j] + edge_cost(i, j, blocks) * x[i, j]
                for i in totes if i != j
            )
        )

    # Linearization of W[i,j] = x[i,j] * F[i]
    for i, j in arcs:
        model.addConstr(W[i, j] <= F_max * x[i, j])
        model.addConstr(W[i, j] >= F[i] - F_max * (1 - x[i, j]))
        model.addConstr(W[i, j] <= F[i])

    # Order completion: C[k] >= F[tote] for every tote belonging to order k
    # Since we minimise sum C[k], this forces C[k] = max F[tote] for totes of order k
    for k, totes_k in order_tote_map.items():
        for t in totes_k:
            if t in blocks:
                model.addConstr(C[k] >= F[t])

    model.optimize()

    # Accept optimal or best incumbent found within time limit
    if model.SolCount == 0:
        raise RuntimeError("Gurobi found no feasible solution within the time limit.")

    solver_used = "gurobi" if model.Status == GRB.OPTIMAL else "gurobi_timelimit"

    start_tote = [i for i in totes if s[i].X > 0.5][0]
    seq = [start_tote]
    cur = start_tote
    visited = {start_tote}
    while True:
        nxt = [j for j in totes if j != cur and x[cur, j].X > 0.5]
        if not nxt:
            break
        cur = nxt[0]
        if cur in visited:
            break
        seq.append(cur)
        visited.add(cur)

    return seq, float(model.ObjVal), solver_used


def solve_item_mip(item_pairs, item_terms, completing_orders):
    """Exact MIP to find the optimal item ordering within a tote.

    completing_orders: list of order_ids whose last tote is the current tote.
    Only these orders contribute to the time component of the objective —
    their completion time equals the finish time of their last item in this tote.
    """
    n = len(item_pairs)
    if n <= 1:
        return item_pairs[:]

    idxs = list(range(n))
    arcs = [(i, j) for i in idxs for j in idxs if i != j]

    completing_set = set(completing_orders)
    # Map each completing order to its item indices in this tote
    order_items = {}
    for idx, (oid, _) in enumerate(item_pairs):
        if oid in completing_set:
            order_items.setdefault(oid, set()).add(idx)

    F_max_item = n * (PLACE_TIME + BIN_SWITCH_TIME)

    model = gp.Model("item_sequence_exact")
    model.Params.OutputFlag = 0
    model.Params.TimeLimit = MIP_TIME_LIMIT

    x = model.addVars(arcs, vtype=GRB.BINARY, name="x")
    s = model.addVars(idxs, vtype=GRB.BINARY, name="s")
    e = model.addVars(idxs, vtype=GRB.BINARY, name="e")
    u = model.addVars(idxs, vtype=GRB.INTEGER, lb=1, ub=n, name="u")

    # F[j] = cumulative local time when item j is placed
    F = model.addVars(idxs, lb=0.0, ub=F_max_item, name="F")
    # W[i,j] linearizes x[i,j] * F[i]
    W = model.addVars(arcs, lb=0.0, ub=F_max_item, name="W")
    # C[co] = local completion time for completing order co
    co_list = list(completing_set)
    C = model.addVars(co_list, lb=0.0, ub=F_max_item, name="C") if co_list else {}

    n_float = float(n)

    def item_edge_cost(i, j):
        return BIN_SWITCH_TIME if item_pairs[i][0] != item_pairs[j][0] else 0.0

    node_optional = gp.quicksum(
        item_terms["node_coeff"][i] * (n_float + 1.0 - u[i]) for i in idxs
    )
    compat_optional = gp.quicksum(
        OPT_EDGE_WEIGHT * x[i, j] for (i, j) in item_terms["compat"]
    )

    time_obj = gp.quicksum(C[co] for co in completing_set) if co_list else 0.0
    model.setObjective(
        time_obj - OPTIONALITY_LAMBDA * (node_optional + compat_optional),
        GRB.MINIMIZE,
    )

    for i in idxs:
        model.addConstr(gp.quicksum(x[j, i] for j in idxs if j != i) + s[i] == 1)
    for i in idxs:
        model.addConstr(gp.quicksum(x[i, j] for j in idxs if j != i) + e[i] == 1)
    model.addConstr(gp.quicksum(s[i] for i in idxs) == 1)
    model.addConstr(gp.quicksum(e[i] for i in idxs) == 1)

    for i, j in arcs:
        model.addConstr(u[i] - u[j] + n * x[i, j] <= n - 1)

    # Item finish time: F[j] = PLACE_TIME + sum_i(W[i,j] + item_edge_cost(i,j)*x[i,j])
    # For start item: sum_i x[i,j] = 0, so F[j] = PLACE_TIME automatically.
    for j in idxs:
        model.addConstr(
            F[j] == PLACE_TIME + gp.quicksum(
                W[i, j] + item_edge_cost(i, j) * x[i, j]
                for i in idxs if i != j
            )
        )

    # Linearization of W[i,j] = x[i,j] * F[i]
    for i, j in arcs:
        model.addConstr(W[i, j] <= F_max_item * x[i, j])
        model.addConstr(W[i, j] >= F[i] - F_max_item * (1 - x[i, j]))
        model.addConstr(W[i, j] <= F[i])

    # Completing order completion times
    for co, item_idxs in order_items.items():
        for idx in item_idxs:
            model.addConstr(C[co] >= F[idx])

    model.optimize()

    # Use best incumbent if available; fallback to order-id sort if no solution found
    if model.SolCount == 0:
        return [item_pairs[i] for i in sorted(idxs, key=lambda k: item_pairs[k][0])]

    start_idx = [i for i in idxs if s[i].X > 0.5][0]
    seq = [start_idx]
    cur = start_idx
    visited = {start_idx}
    while True:
        nxt = [j for j in idxs if j != cur and x[cur, j].X > 0.5]
        if not nxt:
            break
        cur = nxt[0]
        if cur in visited:
            break
        seq.append(cur)
        visited.add(cur)

    return [item_pairs[i] for i in seq]


def build_sorter_input(seq, blocks):
    cols = {0: "circle", 1: "pentagon", 2: "trapezoid", 3: "triangle",
            4: "star", 5: "moon", 6: "heart", 7: "cross"}

    rows = {}
    for tote in seq:
        conv = ((blocks[tote]["first_bin"] - 1) % NUM_CONVEYORS) + 1
        rows.setdefault(conv, {name: 0 for name in cols.values()})
        for shape in blocks[tote]["items"]:
            if shape in cols:
                rows[conv][cols[shape]] += 1

    out = []
    for conv in sorted(rows.keys()):
        r = {"conv_num": conv}
        r.update(rows[conv])
        out.append(r)
    return out


def build_item_offload(seq, blocks):
    rows = []
    pos = 0
    for tote in seq:
        for item_type in blocks[tote]["items"]:
            pos += 1
            rows.append({"sequence_pos": pos, "item_type": item_type})
    return rows


def write_csv(path, fieldnames, rows):
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)


all_bases = _get_input_bases(RUN_ID)
if not all_bases:
    raise RuntimeError("No input runs found. Ensure inputs/runs exists or set RUN_ID appropriately.")

aggregate = []

for base in all_bases:
    INPUT_ITEMTYPES = base / "order_itemtypes.csv"
    INPUT_QUANTITIES = base / "order_quantities.csv"
    INPUT_TOTES = base / "orders_totes.csv"

    tasks = build_tasks()
    blocks = build_tote_blocks(tasks)
    order_tote_map = build_order_tote_map(blocks)
    terms = build_optionality_terms(blocks)

    seq, objective_score, solver_used = solve_exact_mip(blocks, terms, order_tote_map)

    # Determine which orders complete at each tote (last tote in sequence for that order)
    seq_pos = {tote: pos for pos, tote in enumerate(seq)}
    orders_completing_in_tote = {tote: [] for tote in blocks}
    for order_id, totes_set in order_tote_map.items():
        in_seq = [t for t in totes_set if t in seq_pos]
        if in_seq:
            last_tote = max(in_seq, key=lambda t: seq_pos[t])
            orders_completing_in_tote[last_tote].append(order_id)

    for tote in blocks:
        item_terms = build_item_optionality_terms(blocks[tote]["item_pairs"])
        completing = orders_completing_in_tote[tote]
        blocks[tote]["items"] = [p[1] for p in solve_item_mip(blocks[tote]["item_pairs"], item_terms, completing)]

    total_time, sum_completion, optionality_score, objective_check = sequence_metrics(
        seq, blocks, terms, order_tote_map
    )

    run_out = OUT / "exact_mip_runs" / base.name
    run_out.mkdir(parents=True, exist_ok=True)

    write_csv(
        run_out / "exact_mip_tote_sequence.csv",
        ["sequence_pos", "tote"],
        [{"sequence_pos": i + 1, "tote": t} for i, t in enumerate(seq)],
    )

    sorter_rows = build_sorter_input(seq, blocks)
    write_csv(
        run_out / "optimized_input_from_exact_mip_model.csv",
        ["conv_num", "circle", "pentagon", "trapezoid", "triangle", "star", "moon", "heart", "cross"],
        sorter_rows,
    )

    write_csv(
        run_out / "exact_mip_tote_item_plan.csv",
        ["sequence_pos", "item_type"],
        build_item_offload(seq, blocks),
    )

    summary_row = {
        "run_name": base.name,
        "solver_used": solver_used,
        "total_time": total_time,
        "sum_order_completion_time": sum_completion,
        "optionality_score": optionality_score,
        "objective_score": objective_check,
        "solver_objective": objective_score,
        "n_totes": len(seq),
        "n_units": sum(b["units"] for b in blocks.values()),
    }

    write_csv(run_out / "exact_mip_summary.csv", list(summary_row.keys()), [summary_row])
    aggregate.append(summary_row)

if RUN_ID == "all":
    write_csv(OUT / "exact_mip_all_runs_summary.csv", list(aggregate[0].keys()), aggregate)
    print(f"Processed {len(aggregate)} runs.")
    print("Wrote aggregate: outputs/exact_mip_all_runs_summary.csv")
else:
    print(f"Solved with: {aggregate[0]['solver_used']}")
    print(f"Exact optimal objective: {aggregate[0]['objective_score']:.3f}")
    print(f"Sum order completion time: {aggregate[0]['sum_order_completion_time']:.3f}")
    print("Wrote run outputs under outputs/")


Restricted license - for non-production use only - expires 2027-11-29
Processed 50 runs.
Wrote aggregate: outputs/exact_mip_all_runs_summary.csv
